In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import os
from runner import DualRunner
import pandas as pd

PG_CONNINFO = (
    f"host=127.0.0.1 "
    f"port={os.getenv("POSTGRES_PORT", 5432)} "
    f"dbname={os.getenv("POSTGRES_DB")} "
    f"user={os.getenv("POSTGRES_USER")} "
    f"password={os.getenv("POSTGRES_PASSWORD")}"
)

runner = DualRunner(
    pg_conninfo=PG_CONNINFO,
    duckdb_path=":memory:"
)

display(runner.run_pg("select version()"))
display(runner.run_dd("select version()"))

## pandas 設定

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

,version
0,PostgreSQL 17.7 (Debian 17.7-3.pgdg13+1) on aa...


,"""version""()"
0,v1.4.3


# データ加工のためのSQL
## 一つの値に対する処理

In [6]:
runner.check("""--sql
drop table if exists access_log;
create table access_log (
    stamp timestamp,
    referrer text,
    url text
);
insert into access_log (stamp, referrer, url) values
('2016-08-26 12:02:00', 'http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1', 'http://www.example.com/video/detail?id=001'),
('2016-08-26 12:02:01', 'http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1', 'http://www.example.com/video#ref'),
('2016-08-26 12:02:01', 'https://www.other.com/', 'http://www.example.com/book/detail?id=002');            
select * from access_log;
             
""")

### ✅ SAME

,stamp,referrer,url
0,2016-08-26 12:02:00,http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video/detail?id=001
1,2016-08-26 12:02:01,http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video#ref
2,2016-08-26 12:02:01,https://www.other.com/,http://www.example.com/book/detail?id=002


In [7]:
runner.pg("""--sql

-- 正規表現を使って値を抽出する
select
    stamp,
    referrer,
    url,
    substring(referrer from 'https?://([^/]*)') as referrer_domain,
    substring(url from '//[^/]+([^?#]+)') as path,
    substring(url from 'id=([^&]*)') as id
from access_log
""")

runner.dd("""--sql

select
    stamp,
    referrer,
    url,
    regexp_extract(referrer, 'https?://([^/]*)', 1) as referrer_domain,
    regexp_extract(url,  '//[^/]+([^?#]+)', 1) as path,
    regexp_extract(url,  'id=([^&]*)', 1) as id
from access_log

""")

### 🐘 PostgreSQL Result

,stamp,referrer,url,referrer_domain,path,id
0,2016-08-26 12:02:00,http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video/detail?id=001,www.other.com,/video/detail,001
1,2016-08-26 12:02:01,http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video#ref,www.other.net,/video,None
2,2016-08-26 12:02:01,https://www.other.com/,http://www.example.com/book/detail?id=002,www.other.com,/book/detail,002


### 🦆 DuckDB Result

,stamp,referrer,url,referrer_domain,path,id
0,2016-08-26 12:02:00,http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video/detail?id=001,www.other.com,/video/detail,001
1,2016-08-26 12:02:01,http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video#ref,www.other.net,/video,
2,2016-08-26 12:02:01,https://www.other.com/,http://www.example.com/book/detail?id=002,www.other.com,/book/detail,002


## 文字列を配列に分解する

`split_part(str, '/', 2)` などと書けば、分割する文字と分割した後にインデックスを指定して抽出できる。duckdb でも同じ関数。




In [20]:
runner.pg("""--sql
-- ulr のパスをスラッシュで分割して階層を抽出する
select
    stamp,
    url,
    split_part(substring(url from '//[^/]+([^?#]+)'), '/', 2) as path_1,
    split_part(substring(url from '//[^/]+([^?#]+)'), '/', 3) as path_2
from access_log
""")

runner.dd("""--sql
-- ulr のパスをスラッシュで分割して階層を抽出する
select
    stamp,
    url,
    split_part(regexp_extract(url, '//[^/]+([^?#]+)', 1), '/', 2) as path_1,
    split_part(regexp_extract(url, '//[^/]+([^?#]+)', 1), '/', 3) as path_2
from access_log
""")

### 🐘 PostgreSQL Result

,stamp,url,path_1,path_2
0,2016-08-26 12:02:00,http://www.example.com/video/detail?id=001,video,detail
1,2016-08-26 12:02:01,http://www.example.com/video#ref,video,
2,2016-08-26 12:02:01,http://www.example.com/book/detail?id=002,book,detail


### 🦆 DuckDB Result

,stamp,url,path_1,path_2
0,2016-08-26 12:02:00,http://www.example.com/video/detail?id=001,video,detail
1,2016-08-26 12:02:01,http://www.example.com/video#ref,video,
2,2016-08-26 12:02:01,http://www.example.com/book/detail?id=002,book,detail


## 日付やタイムスタンプを扱う


In [46]:
runner.check("""--sql
select
    current_date as today,

    -- これはTZ付きデータになる
    current_timestamp as now_with_tz,

    -- localtimestamp や TZを指定すると、TZ無しデータになる
    localtimestamp as local_now,
    current_timestamp at time zone 'UTC' as now_utc,
    current_timestamp at time zone 'Asia/Tokyo' as now_tokyo,

    -- current_timestamp の代わりに now() を使っても同じ
    now() as now_with_tz_2,
    now() at time zone 'UTC' as now_utc_2,
    now() at time zone 'Asia/Tokyo' as now_tokyo_2
""")

### ✅ SAME

,today,now_with_tz,local_now,now_utc,now_tokyo,now_with_tz_2,now_utc_2,now_tokyo_2
0,2026-01-19,2026-01-19 23:32:41.797977+09:00,2026-01-19 23:32:41.797977,2026-01-19 14:32:41.797977,2026-01-19 23:32:41.797977,2026-01-19 23:32:41.797977+09:00,2026-01-19 14:32:41.797977,2026-01-19 23:32:41.797977


In [ ]:
runner.check("""--sql
select
    '2016-08-26 12:02:00+09'::timestamptz as ts_with_tz,
    '2016-08-26 12:02:00'::timestamp as ts_without_tz,
    '2016-08-26'::date as only_date,

    -- cast で変換することも可能
    cast('2016-08-26 12:02:00+09' as timestamptz) as ts_with_tz_cast,
    cast('2016-08-26 12:02:00' as timestamp) as ts_without_tz_cast,
    cast('2016-08-26' as date) as only_date_cast,

    -- 以下の書き方もできる
    timestamptz '2016-08-26 12:02:00+09' as ts_with_tz_literal,
    timestamp '2016-08-26 12:02:00' as ts_without_tz_literal,
    date '2016-08-26' as only_date_literal
""")

### ✅ SAME

,ts_with_tz,ts_without_tz,only_date,ts_with_tz_cast,ts_without_tz_cast,only_date_cast,ts_with_tz_literal,ts_without_tz_literal,only_date_literal
0,2016-08-26 12:02:00+09:00,2016-08-26 12:02:00,2016-08-26,2016-08-26 12:02:00+09:00,2016-08-26 12:02:00,2016-08-26,2016-08-26 12:02:00+09:00,2016-08-26 12:02:00,2016-08-26


### 日付・時刻から特定のフィールドを取り出す


In [74]:
runner.check("""--sql
             
with t as (
    select 
        '2016-08-26 12:02:39'::timestamp as stamp,
        '2016-08-26 12:02:39.82943'::timestamp as stamp_ms
        
)

select
    stamp,
    stamp_ms,
    extract(year from stamp) as year,
    extract(month from stamp) as month,
    extract(day from stamp) as day,
    extract(hour from stamp) as hour,
    extract(minute from stamp) as minute,

    -- second のあつかいが PG と duckdb で異なる
    -- second は duckb では小数点以下を切り捨てるが、PG では小数点以下も含む
    extract(second from stamp) as second, 
    extract(second from stamp_ms) as second_ms,

    extract(microsecond from stamp_ms) as microsecond 
from t
             
             
""")

## ❌ DIFF Detected

#### 🐘 PostgreSQL Result

,stamp,stamp_ms,year,month,day,hour,minute,second,second_ms,microsecond
0,2016-08-26 12:02:39,2016-08-26 12:02:39.829430,2016,8,26,12,2,39.000000,39.829430,39829430


#### 🦆 DuckDB Result

,stamp,stamp_ms,year,month,day,hour,minute,second,second_ms,microsecond
0,2016-08-26 12:02:39,2016-08-26 12:02:39.829430,2016,8,26,12,2,39,39,39829430


PG dtypes:
stamp          datetime64[ns]
stamp_ms       datetime64[ns]
year                   object
month                  object
day                    object
hour                   object
minute                 object
second                 object
second_ms              object
microsecond            object
dtype: object

DuckDB dtypes:
stamp          datetime64[us]
stamp_ms       datetime64[us]
year                    int64
month                   int64
day                     int64
hour                    int64
minute                  int64
second                  int64
second_ms               int64
microsecond             int64
dtype: object


## 欠損値をデフォルト値に置き換える

In [75]:
runner.check("""--sql 
drop table if exists purchase_log_with_coupon;
create table purchase_log_with_coupon (
    purchase_id integer,
    amount integer,
    coupon integer
);
insert into purchase_log_with_coupon (purchase_id, amount, coupon) values
(10001, 3280, null),
(10002, 4650, 500),
(10003, 3870, null);
select * from purchase_log_with_coupon;
             
""")

### ✅ SAME

,purchase_id,amount,coupon
0,10001,3280,NaN
1,10002,4650,500.0
2,10003,3870,NaN


In [77]:
#* 購入額から割引クーポンを引いて、実際の支払額を計算する
runner.check("""--sql
select
    purchase_id,
    amount,
    coupon,
    amount - coupon as actual_payment_pg, --null を四則演算すると null になるので注意
    amount - coalesce(coupon, 0) as actual_payment -- colalesce で null を 0 に変換してから計算する
from purchase_log_with_coupon
""")

### ✅ SAME

,purchase_id,amount,coupon,actual_payment_pg,actual_payment
0,10001,3280,NaN,NaN,3280
1,10002,4650,500.0,4150.0,4150
2,10003,3870,NaN,NaN,3870
